In [1]:
#!pip install seaborn
#!pip install openpyxl
#!pip install sklearn
#!pip install yfinance 

In [2]:
#pip install scikit-learn tensorflow statsmodels

In [3]:
import pandas as pd
import numpy as np
import random as rd
import time
import csv
import seaborn as sbs
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import train_test_split ,GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error


In [4]:
path_name_results='../results/'
file_result = 'Result_MLP_TSLA_stock_prices.csv'

In [5]:
# Load CSV IBM stock prices
df_raw = pd.read_csv('../datasets/TSLA_stock_prices.csv')

# show columns
print("Cols availables:", df_raw.columns.tolist())

# create dataset
dataset = pd.DataFrame()
dataset['date'] = pd.to_datetime(df_raw['Date'])
dataset['num_observations'] = df_raw['Close']  


Cols availables: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              4009 non-null   datetime64[ns]
 1   num_observations  4009 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 62.8 KB


In [7]:
#checks if there are null variables
dataset.isna().sum()

date                0
num_observations    0
dtype: int64

In [8]:
def salvar_resultado(nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration):
  #Script to write training cycle results
  data = [nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration]
  fields = ['Dataset','Best Params','n_time_steps','MSE', 'RMSE', 'MAE','MAPE','sMAPE','Duration']
  with open(f'{path_name_results}{file_result}', "a",newline='') as csv_file:
    writer = csv.writer(csv_file,delimiter=';')
    writer.writerow(data)  
    csv_file.close()
  print(fields)
  print(data)
    
#Script to create the results file
def criar_arquivo_resultado():
  fields = ['Dataset','Best Params','n_time_steps','MSE', 'RMSE', 'MAE','MAPE','sMAPE','Duration']
  with open(f'{path_name_results}{file_result}', "w",newline='') as csv_file:
    writer = csv.writer(csv_file,delimiter=';')
    writer.writerow(fields)    

In [9]:
def previsao_MLP(nm_dataset, dataset, n_time_steps):
    
    # dataframe treatment
    df = pd.DataFrame()
    df['num_observations'] = dataset['num_observations']  
    
    # time series transform - shift n_time_steps
    if n_time_steps > 0:
        for n_step in range(1, n_time_steps + 1, 1):
            df['vl-' + str(n_step)] = dataset['num_observations'].shift(n_step)
    # if n_time_steps == 0, no lag features are created
    
    df.dropna(inplace=True)
    
    # Split dataset into train (80%) and test (20%)
    nlinhas = int(np.round(df.shape[0] * 0.80))
    
    max_size_train_split = int(np.round(nlinhas / 5)) 
    max_size_test_split = int(np.round((df.shape[0] - nlinhas) / 5))
    size_split = 5
    
    # Handle X_train and X_test for n_time_steps = 0
    if n_time_steps > 0:
        X_train = df.iloc[0:nlinhas, 1:1 + n_time_steps]
        X_test = df.iloc[nlinhas:dataset.shape[0], 1:1 + n_time_steps]
    else:
        # For n_time_steps = 0, use a constant feature
        X_train = pd.DataFrame(np.ones((nlinhas, 1)), columns=['constant'])
        X_test = pd.DataFrame(np.ones((dataset.shape[0] - nlinhas, 1)), columns=['constant'])
    
    y_train = df.iloc[0:nlinhas, 0].values
    y_test = df.iloc[nlinhas:dataset.shape[0], 0].values
    
    scaler_X = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    
    # Scale features
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    
    # Scale target
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
    # Keep original y_test for inverse transform later
    
    # Stores the training execution start time
    Hora_Inicio = time.time()
    
    # Cross-validated for time series
    ts_cv = TimeSeriesSplit(
        n_splits=size_split,
        max_train_size=max_size_train_split,
        gap=2,
        test_size=max_size_test_split,
    )
    
    param_grid = {
        'hidden_layer_sizes': [(4,6,1), (2,6,1), (6,12,1), (6,18,1)],
        'max_iter': [500],
        'activation': ['relu', 'identity'],
        'solver': ['adam'],
        'alpha': [0.0001, 0.001, 0.01],
    }
    
    modelo = MLPRegressor(random_state=0)
    
    grid = GridSearchCV(modelo, param_grid, n_jobs=-1, 
                        scoring='neg_mean_absolute_percentage_error', 
                        cv=ts_cv, verbose=1)
    
    # Train on scaled data
    grid.fit(X_train_scaled, y_train_scaled)
    
    resultado = str(grid.best_params_)
    
    # Predict on scaled test data
    predict_scaled = grid.predict(X_test_scaled)
    
    # Inverse transform predictions to original scale
    predict = scaler_y.inverse_transform(predict_scaled.reshape(-1, 1)).ravel()
    
    # Stores the training execution end time
    Hora_Fim = time.time()
    
    # Calculate the duration of the training execution
    Duracao = Hora_Fim - Hora_Inicio
    
    # Calculate metrics with original scale values
    # Mean Squared Error
    MSE = mean_squared_error(y_test, predict)
    
    # Square Root of Mean Error - RMSE
    RMSE = np.sqrt(mean_squared_error(y_test, predict))
    
    # Mean Absolute Error - MAE
    MAE = median_absolute_error(y_pred=predict, y_true=y_test)
    
    # Calculate MAPE (Mean Absolute Percentage Error) with protection
    MAPE = ((np.mean(np.abs(y_test - predict) / (y_test + 1e-10)))) * 100
    
    # Calculate sMAPE (Symmetric Mean Absolute Percentage Error)
    sMAPE = round(
        np.mean(
            np.abs(predict - y_test) /
            ((np.abs(predict) + np.abs(y_test)) + 1e-10)
        ) * 100, 2
    )
    
    salvar_resultado(nm_dataset, resultado, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duracao)

In [10]:
#create file to results
criar_arquivo_resultado()

print('forecast for TSLA Stock prices')
for n_time_steps in range(0,25): #predict with 1 to 24 past values of medition
    grid = previsao_MLP('TSLA', dataset, n_time_steps)

forecast for TSLA Stock prices
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (6, 18, 1), 'max_iter': 500, 'solver': 'adam'}", 0, 63336.94222470874, 251.66831788031791, 203.30238011663042, 77.57148759815303, 63.9, 6.14958119392395]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (6, 18, 1), 'max_iter': 500, 'solver': 'adam'}", 1, 108.82195310275769, 10.431776124071954, 5.561066523351812, 2.61370849497255, 1.31, 3.125645637512207]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (4, 6, 1), 'max_iter': 500, 'solver': 'adam'}", 2, 141.30216384571114, 11.887058670912293, 6.605164959784588, 3.0283360691311394, 1.52, 3.50707745552063]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (4, 6, 1), 'max_iter': 500, 'solver': 'adam'}", 3, 162.01869861737615, 12.728656591226592, 6.950068597444044, 3.178522230353236, 1.61, 3.2211287021636963]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


...

['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (2, 6, 1), 'max_iter': 500, 'solver': 'adam'}", 20, 921.6119525959855, 30.358062398578497, 19.48229628878505, 8.368614485078568, 4.19, 3.164799690246582]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.01, 'hidden_layer_sizes': (6, 12, 1), 'max_iter': 500, 'solver': 'adam'}", 21, 365.88264942234036, 19.12805921734718, 10.586215160795192, 4.9804770906422355, 2.5, 2.227825164794922]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.01, 'hidden_layer_sizes': (4, 6, 1), 'max_iter': 500, 'solver': 'adam'}", 22, 321.1227266301515, 17.919897506128528, 10.260763708411218, 4.718458959892715, 2.37, 3.4504635334014893]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.01, 'hidden_layer_sizes': (6, 18, 1), 'max_iter': 500, 'solver': 'adam'}", 23, 764.3504030202149, 27.646887763728756, 16.256577015246023, 7.452674767860647, 3.74, 2.045793056488037]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['TSLA', "{'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (6, 18, 1), 'max_iter': 500, 'solver': 'adam'}", 24, 947.4347409697747, 30.78042788802285, 18.67788425953114, 8.270292862842933, 4.16, 5.957223415374756]
